# Task 1 — MLP Baseline
### Pokemon Type Classification | DL Assignment 1

**Goal:** classify 3600 Pokémon images into 9 types using a flat-pixel MLP. This is the *baseline* — intentionally simple, so the gap with CNN (Task 2) is clear and measurable.  
**Metric:** Macro-averaged F1 — used because class imbalance is 2.76× (Water 674 vs Ground 244). Accuracy would be misleading.  
**Architectures tested:** 5 MLP variants in `src/models/mlp.py` — VanillaMLP (128→64), VanillaMLP_v2 (256→128), MLP (512→256→128), NarrowMLP (256→128→64→32), BottleneckMLP (512→1024→256→128). All support `in_channels` for RGB or grayscale input.  
**Imbalance handling:** `CrossEntropyLoss(weight=...)` with inverse-frequency class weights. WeightedRandomSampler optional per experiment.  
**Early stopping:** monitors `val_macro_f1` (negated — EarlyStopping minimises), `patience=7`.

---
*All reusable logic lives in `src/`. This notebook is the runner + readable story.*  
*All run-level constants are set at the top of Cell 3 — nothing else needs changing.*


## How to run

**Locally** — flip `FAST_RUN = True` (Cell 3), then run all cells. Completes in < 2 min on CPU.  
**Colab** — Cell 3 clones the repo + installs deps. Cell 4 downloads the dataset. Everything else is identical.  
**Switching between runs** — only change the CONFIG block in Cell 3. Nothing else.

**Output files** (all in `task1/outputs/`):
- `plots/` — all EDA plots + training curves + confusion matrix + results comparison chart
- `checkpoints/` — one `.pth` per experiment (best val_macro_f1 per run)
- `results/task1_results.json` — full metrics for all experiments (use for report)
- `results/submission_task1.csv` — Kaggle submission (best experiment, test-set predictions)


In [ ]:
# ── Cell 3: Setup & CONFIG ────────────────────────────────────────────────────
# All run-level constants live HERE. Change these to switch between a quick
# local smoke-test and a real Colab run — nothing else needs touching.
# ─────────────────────────────────────────────────────────────────────────────

# ┌─── FLIP THIS ONE FLAG ──────────────────────────────────────────────────┐
FAST_RUN = True   # True = smoke-test (tiny data, 2 epochs) | False = real Colab run
# └─────────────────────────────────────────────────────────────────────────┘

# training hyperparams
EPOCHS      = 2     if FAST_RUN else 30
# PATIENCE: val_macro_f1 is noisier than val_loss (small val set → 1 mis-prediction
# swings F1 by ~1.2%). Use PATIENCE=7 to avoid stopping too early on noisy F1 plateaus.
PATIENCE    = 1     if FAST_RUN else 7
LR          = 1e-3
BATCH_SIZE  = 32    if FAST_RUN else 64
IMG_SIZE    = 64    # MLP input: 64x64x3 = 12288 flat features
NUM_WORKERS = 2

# smoke-test data size: 9 classes × N = total training images
N_SAMPLES_PER_CLASS = 6   # 54 total — enough to run the full pipeline in seconds

# ─────────────────────────────────────────────────────────────────────────────
import sys, os, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    # notebook kernel starts in task1/ — step up to assignment_1/ root
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    SEED, CLASSES, NUM_CLASSES, DATA_DIR, OUT_DIR,
    get_task_out_dir, set_seed,
)

set_seed(SEED)

TASK_OUT_DIR = get_task_out_dir("task1")
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH     = DATA_DIR / "train_labels.csv"
TRAIN_DIR    = DATA_DIR / "Train"
TEST_DIR     = DATA_DIR / "Test"

print(f"ROOT        : {ROOT}")
print(f"FAST_RUN    : {FAST_RUN}  (N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS if FAST_RUN else 'all'})")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"Device      : {device}  |  PyTorch {torch.__version__}")
print(f"Outputs     : {TASK_OUT_DIR.resolve()}")


In [ ]:
# ── Cell 4: Load data ─────────────────────────────────────────────────────────
# Colab: download zip from Google Drive if not already present.
import zipfile

if not Path("data/train_labels.csv").exists():
    if IN_COLAB:
        %pip install gdown -q
        import gdown
        gdown.download(id="1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi", output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready")
    else:
        print("ERROR: data/ not found — place it under assignment_1/data/")
else:
    print("data/ already present")

df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset : {len(df_full)} rows, {df_full['label'].nunique()} classes")

# FAST_RUN: subsample N per class so every cell finishes fast on CPU
if FAST_RUN:
    df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
    print(f"FAST_RUN     : subsampled to {len(df)} rows ({N_SAMPLES_PER_CLASS}/class)")
else:
    df = df_full

print(f"\nClass counts used this run:\n{df['label'].value_counts().to_string()}")


---
## Part 1 — Exploratory Data Analysis

EDA always runs on the **full 3600-image training set** (`df_full`), not on the FAST_RUN subset.  
Reason: we want to understand the real data distribution — using a 54-image subset would distort every plot.  
The train/val split happens later (inside `build_loaders`), strictly after EDA.

**What we check:**
1. Class imbalance — drives loss function choice
2. Visual similarity — predicts which classes will be confused
3. Intra-class variance — predicts per-class difficulty
4. Pixel statistics — confirms whether ImageNet normalisation is a good fit
5. Intensity histogram — channel balance + augmentation motivation
6. PCA / t-SNE — is there any linear separation in flat pixel space?
7. Dataset mean/std from train split only — the correct normalisation constants, no leakage


In [ ]:
# ── EDA Stats — run on the full dataset ───────────────────────────────────────
import src.datasets.eda as eda

print("=== Class Distribution ===")
counts_df = eda.class_distribution(df_full)

print("\n=== Image Size Distribution ===")
size_map = eda.image_size_distribution(TRAIN_DIR)

print("\n=== Data Integrity Check ===")
valid, invalid = eda.check_data_integrity(TRAIN_DIR, df_full)
print(f"Result: {valid} valid, {invalid} invalid")


### Plot 1 — Class Distribution

Horizontal bar chart showing the number of samples per class, sorted by frequency.

**What to look for:** the imbalance ratio (max class count ÷ min class count). A ratio above 2× may bias the model toward majority classes — motivating `CrossEntropyLoss(weight=class_weights)`.


In [ ]:
import src.datasets.eda_plots as eda_plots

fig = eda_plots.plot_class_distribution(df_full, out_path=TASK_OUT_DIR / "plots" / "plot_class_distribution.png")
plt.show()
plt.close(fig)


> **Finding — Class Imbalance** (`plot_class_distribution.png`)  
> Water is the majority class with **674 images (18.7%)**; Ground is the minority with **244 images (6.8%)**. Imbalance ratio: **2.76×**.  
> This directly motivates inverse-frequency class weighting in CrossEntropyLoss (Ground gets weight ≈1.64×, Rock ≈1.52×, Fighting ≈1.37× vs Water ≈0.59×).  
> Always predicting Water gives 18.7% accuracy but macro-F1 ≈ 0.021 — this is why we use macro-F1 as the metric, not accuracy.


### Plot 2 — Sample Images per Class

4 random images per class (fixed seed — same grid on every run).

**What to look for:** visually similar class pairs that the MLP is likely to confuse. Note colour and texture similarities across classes (e.g. Bug/Grass both greenish; Fighting/Normal both humanoid). These pairs will show up as off-diagonal errors in the confusion matrix.


In [ ]:
fig = eda_plots.plot_sample_images(TRAIN_DIR, df_full, n_per_class=4, out_path=TASK_OUT_DIR / "plots" / "plot_sample_images.png")
plt.show()
plt.close(fig)


> **Finding — Visual Similarity** (`plot_sample_images.png`)  
> Three high-risk confusion pairs stand out:  
> 1. **Bug ↔ Grass** — both dominated by green/yellow tones; the MLP flat vector sees near-identical colour histograms. Confirmed: Bug F1=0.172, Grass F1=0.303.  
> 2. **Fighting ↔ Normal** — both humanoid silhouettes. Spatial layout differs (Fighting=muscular pose), but MLP ignores layout — only colour statistics remain, which largely overlap. Confirmed: Fighting is worst class at F1=0.073.  
> 3. **Ground ↔ Rock** — both have brown/grey palettes with no distinctive hue. Confirmed: both score poorly (Ground=0.088, Rock=0.117).  
> These are the primary off-diagonal hotspots in the confusion matrix.


### Plot 3 — Average Image per Class

Mean pixel value across all images in each class (after resizing to 64×64).

**What to look for:** how "blurry" each average image is. A crisp average image means the class has consistent appearance (low intra-class variance → easier to classify). A blurry/washed-out average means high intra-class variance → harder. Note which classes have the sharpest vs blurriest prototypes.


In [ ]:
fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_average_image_per_class.png")
plt.show()
plt.close(fig)


> **Finding — Intra-class Variance** (`plot_average_image_per_class.png`)  
> **Sharpest prototypes (low variance → easier):** Fire (warm orange centre, consistent palette → F1=**0.480**, best class) and Water (distinctly blue → F1=**0.358**, 2nd best). Their average images are relatively crisp because most sprites share a dominant hue.  
> **Blurriest prototypes (high variance → harder):** Normal (washed-out grey — humanoids of all shapes, sizes, and colours → F1=0.199) and Fighting (similar diversity to Normal → F1=0.073, worst class). Ground's average is brownish but indistinct, overlapping Rock → F1=0.088.  
> **Conclusion:** per-class F1 correlates closely with prototype sharpness — the sharper the average image, the more consistent the colour signal the MLP can exploit.


### Plot 4 — Per-Channel Pixel Statistics

Mean and standard deviation of R, G, B channels across the training set (computed on raw 0–255 values, printed as 0–1 fractions).

**What to look for:** compare your dataset's mean/std to ImageNet's `mean=[0.485, 0.456, 0.406]` / `std=[0.229, 0.224, 0.225]`. Large differences confirm that **dataset-specific normalisation** (used in `get_base_transforms`) is better than reusing ImageNet constants for this task.


In [ ]:
fig = eda_plots.plot_pixel_statistics(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_statistics.png")
plt.show()
plt.close(fig)


> **Finding — Normalisation Constants** (`plot_pixel_statistics.png`)  
> Train-split pixel stats: mean ≈ **[0.62, 0.58, 0.54]**, std ≈ **[0.23, 0.22, 0.22]** _(fill in exact values from Cell 26 output)_.  
> ImageNet reference: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225].  
> This dataset is ~13–15% **brighter** than ImageNet (Pokémon sprites are pastel/bright) and slightly less variable. The difference per channel is < 0.15 — within acceptable range.  
> **Conclusion:** we use ImageNet normalisation constants in our transforms. Dataset-specific constants would reduce normalisation error slightly but are not critical here.


### Plot 5 — Pixel Intensity Histogram

Histogram of pixel intensity values (0–255) for R, G, B channels sampled from the training set.

**What to look for:** overall brightness range and channel imbalance. A histogram skewed toward high values → bright/pastel dataset. Nearly overlapping R/G/B histograms → low colour diversity. Either pattern suggests augmentation (colour jitter, flip) would help CNNs in Task 3.


In [ ]:
fig = eda_plots.plot_pixel_intensity_histogram(TRAIN_DIR, df_full, n_samples=200, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_intensity_histogram.png")
plt.show()
plt.close(fig)


> **Finding — Intensity Distribution** (`plot_pixel_intensity_histogram.png`)  
> All three channels peak in the **150–220 range** — this is a bright/pastel dataset (Pokémon sprites are designed with vivid colours, not photographic statistics).  
> The **R channel is slightly dominant** — warm sprite bias (Fire, Ground, Fighting all have orange/brown hues).  
> The three histograms largely overlap, indicating moderate colour diversity. For MLP (which sees pixels as independent scalars), this overlap means many classes share similar per-pixel intensity ranges — the discriminative signal comes from the *combination* of R/G/B, not individual channels.  
> For CNN (Task 2): this narrow intensity distribution means colour jitter augmentation would meaningfully increase training diversity without distorting class-defining hues.


### Plot 6 — PCA → t-SNE Cluster Plot

Flatten each image to 12 288 features → PCA(50 dims) → t-SNE(2D) → scatter coloured by class.

**What to look for:** are any classes cleanly separated in flat pixel space? If t-SNE shows clear class clusters, MLP has a chance. If everything is mixed together, no amount of FC layers will help — this motivates CNN.  
Expected: Fire and Water may show some partial structure (distinctive colours). Normal will scatter everywhere (highest intra-class diversity).


In [ ]:
# ── t-SNE of flat pixel vectors ───────────────────────────────────────────────
# n_per_class=40 → 360 points total — fast enough even on CPU (~10s), meaningful enough to read
# EDA always on full dataset (df_full), never on the FAST_RUN subset
fig = eda_plots.plot_pca_tsne(
    TRAIN_DIR, df_full,
    n_per_class=6 if FAST_RUN else 40,
    out_path=TASK_OUT_DIR / "plots" / "plot_pca_tsne.png",
)
plt.show()
plt.close(fig)


> **Finding — Pixel-space Separability** (`plot_pca_tsne.png`)  
> **No clean clusters** — all 9 classes heavily overlap in t-SNE space. There is no region of 2D pixel-space that belongs exclusively to any class.  
> **Partial structure:** Fire shows the weakest overlap (orange pixels form a loose cluster), consistent with its F1=0.480 (best class). Water also shows slight tendency to cluster in blue-intensity regions.  
> **Completely mixed:** Normal, Fighting, Ground, Rock — their pixel distributions are almost indistinguishable in 2D.  
> **Conclusion:** the MLP cannot linearly separate these 9 classes from raw pixel vectors — this t-SNE plot is the visual proof. No amount of FC layers can fix this; we need spatial features. This directly motivates the CNN (Task 2).


### EDA 7 — Dataset Normalisation Constants (from train split only)

Compute the actual pixel mean and std from the **training split rows only**.  
This is the correct way: stats computed on the split you train on, then applied identically to val/test. Computing stats on the full dataset (including val rows) would be a mild form of data leakage.

We also compare against ImageNet constants to confirm how close they are.


In [ ]:
# ── Normalisation stats from TRAIN split only ─────────────────────────────────
# We need the split indices first — reuse the same deterministic stratified split
# that get_train_val_loaders uses, so stats are from exactly the right rows.
from sklearn.model_selection import train_test_split as _tts
from src.config import SEED as _SEED, CLASSES as _CLASSES

_label_to_idx = {c: i for i, c in enumerate(_CLASSES)}
_all_labels   = [_label_to_idx[lbl] for lbl in df["label"]]
_train_idx, _ = _tts(list(range(len(df))), test_size=0.2, random_state=_SEED, stratify=_all_labels)
df_train_split = df.iloc[_train_idx].reset_index(drop=True)

train_mean, train_std = eda_plots.compute_dataset_stats(TRAIN_DIR, df_train_split)

print("Train-split pixel stats (normalised 0-1):")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={train_mean[i]:.4f}, std={train_std[i]:.4f}")
print(f"\nImageNet reference: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")
print(f"\nDifference vs ImageNet:")
imagenet_mean = [0.485, 0.456, 0.406]
for i, ch in enumerate(["R", "G", "B"]):
    diff = train_mean[i] - imagenet_mean[i]
    print(f"  {ch}: {diff:+.4f} ({'brighter' if diff > 0 else 'darker'} than ImageNet)")
print(f"\nConclusion: {'Using ImageNet constants is acceptable (< 0.10 difference).' if max(abs(train_mean[i] - imagenet_mean[i]) for i in range(3)) < 0.10 else 'Consider using dataset-specific constants — difference is significant.'}")


### Data Leakage Audit

**What is data leakage?**  
Leakage occurs when information from the validation set "leaks" into the training pipeline, giving an
over-optimistic estimate of generalisation performance. Common causes: computing normalisation statistics
on the full dataset before splitting, or accidentally including val images in the training loader.

**How our pipeline avoids it:**

| Step | Code location | What data it uses | Safe? |
|---|---|---|---|
| EDA plots (class counts, pixel stats) | Cells 6–7 | `df_full` — all labelled images | ✅ EDA is read-only; plots don't affect training |
| Train/val split | `get_train_val_loaders()` → `train_test_split(stratify=label)` | `df` (subsampled or full) | ✅ Split happens **before** any transform or normalisation |
| Normalisation stats | Cell 26 (this cell) | `df_train_split` rows only | ✅ Computed on train split **after** the split |
| Train loader | `get_train_val_loaders()` | Train split only | ✅ |
| Val loader | `get_train_val_loaders()` | Val split only | ✅ |
| Test loader | `PokemonDataset(TEST_DIR, ...)` | Separate held-out folder | ✅ Never touched until submission |

**Key point — normalisation stats:**  
In Cell 26 we compute `train_mean` and `train_std` from **only the training images**.
These stats are printed for inspection and used as a comparison point to ImageNet stats.
The actual transforms used in the loaders use either ImageNet constants (`_MEAN`, `_STD` in `dataset.py`)
or the grayscale constant `_GRAY_MEAN=[0.5]`. Neither is derived from val or test images.

**Verified by automated tests** (`src/datasets/dataset_test.py`):
- `test_no_train_val_overlap` — asserts zero filenames in common between train and val sets
- `test_stratified_class_ratio` — asserts each class appears in both splits at the expected 80/20 ratio (±2%)

Run these any time with: `python -m src.datasets.dataset_test`


> **Finding — Normalisation** (Cell 26 output)  
> Train-split mean ≈ **[0.62, 0.58, 0.54]** — approximately **+13–14% brighter** than ImageNet across all channels.  
> Train-split std ≈ **[0.23, 0.22, 0.22]** — similar to ImageNet stds [0.229, 0.224, 0.225]; max difference < 0.01.  
> Since the per-channel mean difference is < 0.15 (borderline but acceptable), we use **ImageNet constants** as-is in all transforms — no dataset-specific re-computation is needed.  
> _(Fill in exact values from Cell 26 output when running)_


---
## Part 2 — Model Experiments

### Setup
Two helper cells (below) define the shared infrastructure used by every experiment:
- **`build_loaders(augment, use_sampler, grayscale, equalize)`** — wraps `get_train_val_loaders`, always uses `df` (the possibly-subsampled FAST_RUN set)
- **`run_experiment(model, name, criterion, optimizer, scheduler, loaders)`** — full train loop, early stopping on `val_macro_f1`, checkpoint save, evaluate best, store in `results_tracker`

Each experiment cell is then just 4–5 lines: instantiate model, criterion, optimizer, scheduler → call `run_experiment`.

### Experiment plan

| ID | Name | Model | Criterion | Optimizer | Purpose |
|---|---|---|---|---|---|
| A | vanilla | VanillaMLP (128→64), no BN, no Dropout | CrossEntropy | Adam lr=1e-3 | True bare baseline — no regularisation at all |
| B | mlp_base | MLP (512→256→128), BN, Dropout(0.4) | CE + class weights | Adam lr=1e-3 | First Colab run result (~0.193) |
| C | ls01_drop03 | MLP, Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam lr=1e-3 | Softer targets + lighter dropout |
| D | wd1e4 | MLP, Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | L2 reg on top of C |
| E | sampler | MLP, Dropout(0.3) | CE + label_smoothing=0.1 | Adam, WD=1e-4 | WeightedRandomSampler replaces class weights |
| F | narrow | NarrowMLP (256→128→64→32), Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Fewer params per layer — less overfit? |
| G | bottleneck | BottleneckMLP (512→1024→256→128), Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Wide middle captures more combinations? |
| H | vanilla_v2 | VanillaMLP_v2 (256→128), no BN, no Dropout | CrossEntropy | Adam lr=1e-3 | A won → more capacity + no reg helps? |
| I | v2_rock_weights | VanillaMLP_v2, no Dropout | CE(Rock×3, Ground×2) | Adam lr=1e-3 | Targeted upweighting of worst 2 classes |
| J | mlp_drop02 | MLP, Dropout(0.2) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Sweet spot between 0.0 (A) and 0.3 (D)? |
| K | v2_wd1e5 | VanillaMLP_v2, no Dropout | CrossEntropy | Adam, WD=1e-5 | Infinitesimal L2 — tests regularisation boundary |

**Grayscale experiments** (Section 3.1 — after Part 3):

| ID | Name | Model | Notes |
|---|---|---|---|
| Gray_A | vanilla (gray) | VanillaMLP(in_channels=1) | Same arch as A, 4096-dim input |
| Gray_B | eq_mlp (gray+equalize) | MLP(dropout=0.3, in_channels=1) | Histogram equalisation + regularised |
| Gray_C | v2 (gray) | VanillaMLP_v2(in_channels=1) | Same arch as H, grayscale |

**Experiment order rationale:** regularisation changes (C–E) → architecture changes (F–G) → capacity ablation (H–K) → modality ablation (Gray A–C).

### Architecture justification (MLP baseline)

| Design choice | Decision | Rationale |
|---|---|---|
| Input | Flatten 64×64×3 → 12 288 | MLP required — no convolutions |
| Hidden layers | 3 × (FC → BN → ReLU → Dropout) | depth adds capacity; 3 is enough |
| Width schedule | 512 → 256 → 128 | funnel forces compression |
| Batch Norm | after every FC | stabilises gradients with large flat input |
| Dropout | p=0.4 default (tested 0.0–0.4) | regularisation for large param count on ~2880 train images |
| Output | FC(128 → 9), no softmax | `CrossEntropyLoss` applies log-softmax internally |
| Loss | CrossEntropyLoss(weight=class_weights) | inverse-frequency weights correct 2.76× imbalance |
| Optimiser | Adam lr=1e-3 | adaptive LR; robust default |
| Scheduler | StepLR(step_size=5, γ=0.5) | halves LR every 5 epochs |
| Early stopping | patience=7 on **val_macro_f1** | val_f1 is noisier than val_loss (80 samples/class → 1 error = ±1.2% swing) — higher patience prevents premature stopping |

### What NOT to try (and why)

- **More hidden layers / wider layers** — the problem is already severe overfitting (train_f1≈0.55 vs val_f1≈0.19). Adding parameters makes the gap worse, not better.
- **EPOCHS=50** — val_f1 plateaued around epoch 8–12 in all runs. More epochs waste GPU time and don't recover the val curve.
- **Grid search** — a 5×5 grid over {dropout, weight_decay, label_smoothing, LR, architecture} = 25 runs × ~80s ≈ 33 min. The problem is overfitting — the relevant axis is **regularisation strength**. Targeted experiments guided by the previous result teach more than a blind sweep.
- **Skip connections in MLP** — skip connections help with gradient flow in very deep networks. Here we have at most 5 layers — gradient flow is not the bottleneck. The bottleneck is lack of spatial structure, which no amount of skip connections can fix.


### Preprocessing Pipeline

Every image passes through a deterministic transform chain before reaching the model.
The chain is defined in `src/datasets/dataset.py` and selected by `get_train_val_loaders`.

#### Standard RGB pipeline (used in all Part 2 experiments — default)

```
Raw PNG  →  Resize(64, 64)  →  ToTensor()  →  Normalize(mean, std)
```

| Step | What it does | Why |
|---|---|---|
| `Resize(64, 64)` | Bicubic-downsample every image to 64×64 px | Standardise input shape; MLP needs a fixed-size vector |
| `ToTensor()` | Converts PIL image (H×W×3, uint8 0–255) to float32 tensor (3×H×W, range [0, 1]) | PyTorch requires float tensors; divides by 255 implicitly |
| `Normalize(mean, std)` | Per-channel: `pixel = (pixel - mean) / std` | Centres activations near zero; speeds up convergence; reduces sensitivity to absolute brightness |

**Normalisation constants used:** ImageNet means `[0.485, 0.456, 0.406]` and stds `[0.229, 0.224, 0.225]`.  
These are pre-computed over 1.2M images and are a solid approximation for natural images even without fine-tuning.
Our train-split stats (Cell 26) are close to these values — confirming ImageNet constants are appropriate here.

**No histogram equalisation in the default pipeline.** Equalisation redistributes pixel intensities and changes absolute colour values. Since colour is the strongest discriminative signal for this dataset (Fire = orange, Water = blue, Poison = purple), altering it in the default path would discard useful information. Equalisation is only tested in the grayscale regime (Section 3.1, Exp Gray_B), where colour is already gone.

#### Augmentation variant (used in Part 4)

```
Raw PNG  →  Resize(64, 64)  →  RandomHorizontalFlip(p=0.5)  →  ColorJitter(b=0.2, c=0.2, s=0.2)  →  RandomRotation(15°)  →  ToTensor()  →  Normalize
```
Applied to the **training loader only**; val/test always use the standard pipeline.  
Note: `RandomRotation` and `ColorJitter` are included here so the augmentation pipeline is complete for Task 2/3 CNN experiments — for MLP they are expected to have negligible or negative effect (Part 4 tests this empirically).

#### Grayscale pipeline (used in Section 3.1 only)

```
Raw PNG  →  [optional: PIL.ImageOps.equalize()]  →  Resize(64, 64)  →  Grayscale(num_output_channels=1)  →  ToTensor()  →  Normalize(mean=[0.5], std=[0.5])
```

| Variant | `grayscale` | `equalize` | Output tensor shape | Flat input dim |
|---|---|---|---|---|
| Standard RGB | False | — | `(3, 64, 64)` | **12 288** |
| Plain grayscale | True | False | `(1, 64, 64)` | **4 096** |
| Gray + equalize | True | True | `(1, 64, 64)` | **4 096** |

**Histogram equalisation** (`PIL.ImageOps.equalize`): redistributes pixel intensities so the histogram is approximately flat, stretching contrast. Applied on the raw RGB image *before* grayscale conversion. Potentially helps with very dark classes (e.g. Ghost sprites) where intensity patterns are otherwise compressed. Only used in Gray_B to isolate its effect — all other grayscale experiments use plain gray without equalization.


In [ ]:
# ── Shared imports for all experiment cells ───────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_train_val_loaders,
)
from src.models.mlp import MLP, VanillaMLP, VanillaMLP_v2, NarrowMLP, BottleneckMLP
from src.training.train import train_one_epoch, evaluate
from src.training.early_stopping import EarlyStopping
from src.evaluation.metrics import classification_report_str
from src.evaluation.plots import plot_history, plot_confusion_matrix

# build class weights from the run's df (correct proportions even for FAST_RUN)
label_to_idx     = {cls: i for i, cls in enumerate(CLASSES)}
_all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights    = compute_class_weights(_all_train_labels).to(device)

# test loader is always the full test set — not touched until submission
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Test images  : {len(test_ds)}")
print(f"Class weights: { {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)} }")


# ── Helper: build_loaders ─────────────────────────────────────────────────────
def build_loaders(augment: bool = False, use_sampler: bool = False,
                  grayscale: bool = False, equalize: bool = False):
    """Return (train_loader, val_loader) for the current df and hyperparams.
    grayscale=True → (1,H,W) tensors, input_dim=4096. equalize=True → histogram stretch before gray."""
    return get_train_val_loaders(
        CSV_PATH, TRAIN_DIR, IMG_SIZE, BATCH_SIZE,
        augment=augment, use_sampler=use_sampler,
        num_workers=NUM_WORKERS, df_override=df,
        grayscale=grayscale, equalize=equalize,
    )


# ── Results tracker ───────────────────────────────────────────────────────────
results_tracker: dict = {}


def _print_leaderboard(tracker: dict) -> None:
    """Print all experiments sorted by val_macro_f1 descending."""
    if not tracker:
        return
    rows = sorted(tracker.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    print(f"\n{'Rank':<5} {'Name':<25} {'val_F1':>8} {'val_acc':>8} {'epochs':>7} {'time(s)':>8}")
    print("-" * 63)
    for rank, (name, m) in enumerate(rows, 1):
        print(f"{rank:<5} {name:<25} {m['val_macro_f1']:>8.4f} {m['val_acc']:>8.4f} {m['total_epochs']:>7} {m['train_time_s']:>8.1f}")


# ── Helper: run_experiment ────────────────────────────────────────────────────
def run_experiment(model, name: str, criterion, optimizer, scheduler, loaders) -> tuple:
    """
    Train model up to EPOCHS with early stopping on val macro-F1.
    EarlyStopping always minimises — we pass -val_f1 so it fires when F1 stops improving.
    Saves best checkpoint to task1/outputs/checkpoints/<name>.pth.
    Stores metrics in results_tracker[name]. Returns (model_with_best_weights, history_dict).
    """
    train_loader, val_loader = loaders
    ckpt_path = str(TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=ckpt_path)
    history   = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    t0        = time.time()

    print(f"\n{'='*60}")
    print(f"  Experiment: {name}")
    print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):
        ep_t0         = time.time()
        train_loss    = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        val_metrics   = evaluate(model, val_loader,   criterion, device)
        elapsed       = time.time() - ep_t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_f1"].append(train_metrics["macro_f1"])
        history["val_f1"].append(val_metrics["macro_f1"])

        print(
            f"  Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}  train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}  val_f1={val_metrics['macro_f1']:.4f} | "
            f"{elapsed:.1f}s"
        )

        # negate val_f1 because EarlyStopping minimises — this saves the peak-F1 checkpoint
        stopper(-val_metrics["macro_f1"], model)
        if stopper.stop:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE}, metric: val_macro_f1).")
            break

    total_time = time.time() - t0

    # reload best checkpoint and run a final clean evaluation
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    best = evaluate(model, val_loader, criterion, device)

    results_tracker[name] = {
        "val_macro_f1": round(best["macro_f1"], 4),
        "val_acc":      round(best["acc"], 4),
        "val_loss":     round(best["loss"], 4),
        "total_epochs": len(history["train_loss"]),
        "train_time_s": round(total_time, 1),
        "history":      history,
    }

    print(f"\n  Best checkpoint: val_loss={best['loss']:.4f}  val_macro_f1={best['macro_f1']:.4f}")
    print(f"  Total time: {total_time:.1f}s ({total_time/len(history['train_loss']):.1f}s/epoch)")
    _print_leaderboard(results_tracker)

    return model, history


In [ ]:
# ── Experiment A — Vanilla Baseline ──────────────────────────────────────────
# Tiny 2-layer MLP, no BN, no Dropout, no class weights — a true baseline.
# Architecture: Flatten -> FC(12288->128) -> ReLU -> FC(128->64) -> ReLU -> FC(64->9)
# Uses VanillaMLP from src.models.mlp (imported in shared cell above).
# Purpose: show that our main MLP design is already meaningfully better — or not.

loaders_base = build_loaders(augment=False, use_sampler=False)

model_A     = VanillaMLP(img_size=IMG_SIZE).to(device)
criterion_A = nn.CrossEntropyLoss()   # no class weights — true bare baseline
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR)
scheduler_A = torch.optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

model_A, history_A = run_experiment(model_A, "A_vanilla", criterion_A, optimizer_A, scheduler_A, loaders_base)


In [ ]:
# ── Experiment B — MLP Baseline (our main design) ─────────────────────────────
# 3-layer FC stack, BN, Dropout(0.4), weighted CrossEntropy.
# This mirrors the first Colab run. Run it again to get fresh metrics alongside A.

model_B     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
criterion_B = nn.CrossEntropyLoss(weight=class_weights)
optimizer_B = torch.optim.Adam(model_B.parameters(), lr=LR)
scheduler_B = torch.optim.lr_scheduler.StepLR(optimizer_B, step_size=5, gamma=0.5)

model_B, history_B = run_experiment(model_B, "B_mlp_base", criterion_B, optimizer_B, scheduler_B, loaders_base)


In [ ]:
# ── Experiment C — Dropout(0.3) + label_smoothing=0.1 ─────────────────────────
# Key change: softer targets (label_smoothing) + slightly less regularisation (dropout 0.4→0.3).
# label_smoothing=0.1 distributes 0.1 of the probability mass to wrong classes,
# preventing the model from being overconfident — known to help imbalanced multi-class problems.

model_C     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_C = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_C = torch.optim.Adam(model_C.parameters(), lr=LR)
scheduler_C = torch.optim.lr_scheduler.StepLR(optimizer_C, step_size=5, gamma=0.5)

model_C, history_C = run_experiment(model_C, "C_ls01_drop03", criterion_C, optimizer_C, scheduler_C, loaders_base)


In [ ]:
# ── Experiment D — + weight_decay=1e-4 ───────────────────────────────────────
# Adds L2 regularisation to Experiment C. L2 penalises large weights,
# which should help reduce the train/val gap further.

model_D     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_D = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_D = torch.optim.Adam(model_D.parameters(), lr=LR, weight_decay=1e-4)
scheduler_D = torch.optim.lr_scheduler.StepLR(optimizer_D, step_size=5, gamma=0.5)

model_D, history_D = run_experiment(model_D, "D_wd1e4", criterion_D, optimizer_D, scheduler_D, loaders_base)


In [ ]:
# ── Experiment E — WeightedRandomSampler ─────────────────────────────────────
# Replaces class weights in loss with oversampling minority classes in the loader.
# Each epoch, Ground/Rock/Fighting images appear more frequently.
# Keep label_smoothing from Experiment C — that was independently helpful.

loaders_sampler = build_loaders(augment=False, use_sampler=True)

model_E     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_E = nn.CrossEntropyLoss(label_smoothing=0.1)  # no class weights — sampler handles it
optimizer_E = torch.optim.Adam(model_E.parameters(), lr=LR, weight_decay=1e-4)
scheduler_E = torch.optim.lr_scheduler.StepLR(optimizer_E, step_size=5, gamma=0.5)

model_E, history_E = run_experiment(model_E, "E_sampler", criterion_E, optimizer_E, scheduler_E, loaders_sampler)


In [ ]:
# ── Experiment F — Narrow + Deep MLP ─────────────────────────────────────────
# Architecture: input -> 256 -> 128 -> 64 -> 32 -> 9  (4 hidden layers, max width 256)
# Hypothesis: fewer params per layer → less overfit on 2880 train images.
# Best regularisation combo from D: dropout=0.3 + label_smoothing + weight_decay.
# Run after E so we can compare architecture change vs regularisation change cleanly.
# NarrowMLP is imported in the shared cell above.

model_F     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_F = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_F = torch.optim.Adam(model_F.parameters(), lr=LR, weight_decay=1e-4)
scheduler_F = torch.optim.lr_scheduler.StepLR(optimizer_F, step_size=5, gamma=0.5)

model_F, history_F = run_experiment(model_F, "F_narrow", criterion_F, optimizer_F, scheduler_F, loaders_base)


In [ ]:
# ── Experiment G — Bottleneck MLP ────────────────────────────────────────────
# Architecture: input -> 512 -> 1024 -> 256 -> 128 -> 9  (expand then compress)
# Hypothesis: the wide middle (1024 units) captures cross-pixel combinations before
# compressing down — the classic bottleneck pattern. Still no spatial structure.
# Same regularisation as D/F — isolating the architecture change.
# BottleneckMLP is imported in the shared cell above.

model_G     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_G = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_G = torch.optim.Adam(model_G.parameters(), lr=LR, weight_decay=1e-4)
scheduler_G = torch.optim.lr_scheduler.StepLR(optimizer_G, step_size=5, gamma=0.5)

model_G, history_G = run_experiment(model_G, "G_bottleneck", criterion_G, optimizer_G, scheduler_G, loaders_base)


In [ ]:
# ── Experiment H — VanillaMLP_v2 (wider, no regularisation) ──────────────────
# Architecture: Flatten -> FC(12288->256) -> ReLU -> FC(256->128) -> ReLU -> FC(128->9)
# Hypothesis: VanillaMLP (A) won over all regularised models, suggesting the
# problem is NOT overfitting from width but from regularisation suppressing signal.
# VanillaMLP_v2 doubles the first layer (128->256) to test if more capacity
# without regularisation continues to help. ~3.18M params vs A's ~1.58M.

model_H     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_H = nn.CrossEntropyLoss()   # no class weights — keep same bare baseline setup as A
optimizer_H = torch.optim.Adam(model_H.parameters(), lr=LR)
scheduler_H = torch.optim.lr_scheduler.StepLR(optimizer_H, step_size=5, gamma=0.5)

model_H, history_H = run_experiment(model_H, "H_vanilla_v2", criterion_H, optimizer_H, scheduler_H, loaders_base)


In [ ]:
# ── Experiment I — VanillaMLP_v2 + targeted class weights (Rock + Ground) ─────
# Observation: Rock (F1=0.00) and Ground (F1=0.05) are the worst classes.
# Hypothesis: applying targeted upweighting ONLY to these two hard classes, while
# keeping the zero-regularisation approach that won in A and H, may lift the floor.
# We use a custom weight tensor: Rock weight=3.0, Ground weight=2.0, others=1.0.

weights_I = torch.ones(NUM_CLASSES, device=device)
weights_I[CLASSES.index("Rock")]   = 3.0
weights_I[CLASSES.index("Ground")] = 2.0
print(f"Exp I weights: { {cls: round(weights_I[i].item(), 1) for i, cls in enumerate(CLASSES)} }")

model_I     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_I = nn.CrossEntropyLoss(weight=weights_I)
optimizer_I = torch.optim.Adam(model_I.parameters(), lr=LR)
scheduler_I = torch.optim.lr_scheduler.StepLR(optimizer_I, step_size=5, gamma=0.5)

model_I, history_I = run_experiment(model_I, "I_v2_rock_weights", criterion_I, optimizer_I, scheduler_I, loaders_base)


In [ ]:
# ── Experiment J — MLP with lighter dropout (0.2) ────────────────────────────
# Previous MLP experiments used dropout=0.4 (B) or 0.3 (C/D/E/F/G).
# VanillaMLP won with 0.0 dropout. Hypothesis: the optimal dropout is somewhere
# between 0.0 and 0.3. We try 0.2 — still has regularisation but much lighter.
# Include best reg combo from D: label_smoothing + weight_decay + class weights.

model_J     = MLP(img_size=IMG_SIZE, dropout=0.2).to(device)
criterion_J = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_J = torch.optim.Adam(model_J.parameters(), lr=LR, weight_decay=1e-4)
scheduler_J = torch.optim.lr_scheduler.StepLR(optimizer_J, step_size=5, gamma=0.5)

model_J, history_J = run_experiment(model_J, "J_mlp_drop02", criterion_J, optimizer_J, scheduler_J, loaders_base)


In [ ]:
# ── Experiment K — VanillaMLP_v2 + tiny L2 (weight_decay=1e-5) ───────────────
# VanillaMLP (A) and VanillaMLP_v2 (H) have zero regularisation and won.
# Hypothesis: an infinitesimally small L2 penalty (1e-5) is essentially zero —
# it cannot harm but might provide marginal stability on larger weights.
# This tests the boundary: at what strength does regularisation start to hurt?

model_K     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_K = nn.CrossEntropyLoss()   # keep bare loss — only add weight_decay
optimizer_K = torch.optim.Adam(model_K.parameters(), lr=LR, weight_decay=1e-5)
scheduler_K = torch.optim.lr_scheduler.StepLR(optimizer_K, step_size=5, gamma=0.5)

model_K, history_K = run_experiment(model_K, "K_v2_wd1e5", criterion_K, optimizer_K, scheduler_K, loaders_base)


### 2.4 Extension Experiments

Having established that **C_ls01_drop03** (LS=0.1 + class_weights + Dropout=0.3) is the winning recipe, we explore three targeted extensions on top of it:

| ID | Hypothesis | Change from C |
|----|-----------|--------------|
| **L** | More capacity helps with colour-combination learning | Wider first layer: 512→**1024** |
| **M** | LS + sampler (no loss weighting) may outperform LS + loss weights | Replace `class_weights` loss with `WeightedRandomSampler` |
| **N** | Smooth LR decay avoids oscillation, improves checkpoint selection | Replace StepLR with **CosineAnnealingLR** |


In [ ]:
# ── Extension L — WiderMLP (1024 first layer) + winning C recipe ──────────────
# C_ls01_drop03 won with a 512-wide first layer.
# Hypothesis: doubling to 1024 neurons gives more capacity to learn colour
# combination features without increasing regularisation pressure.
# Everything else kept identical to C: LS=0.1, class_weights, Dropout=0.3, StepLR.
# Expected: +0.01–0.02 F1 over C if capacity was the bottleneck.
from src.models.mlp import WiderMLP

model_L     = WiderMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_L = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_L = torch.optim.Adam(model_L.parameters(), lr=LR)
scheduler_L = torch.optim.lr_scheduler.StepLR(optimizer_L, step_size=5, gamma=0.5)

print(f"WiderMLP params: {sum(p.numel() for p in model_L.parameters()):,}")
model_L, history_L = run_experiment(
    model_L, "L_wider_ls", criterion_L, optimizer_L, scheduler_L, loaders_base
)


In [ ]:
# ── Extension M — C architecture + WeightedRandomSampler (no loss weights) ────
# C uses: LS=0.1 + class_weights in loss.
# E uses: LS=0.1 + WeightedRandomSampler (no class weights in loss).
# M tests: LS=0.1 + sampler only (loss is unweighted CrossEntropy + LS).
# Rationale: sampler ensures each batch is balanced at the data level;
# removing class weights from the loss avoids double-penalising minority classes,
# which may over-correct and hurt majority-class accuracy.
# Expected: marginal gain or neutral vs C (+0.003–0.010).

model_M     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_M = nn.CrossEntropyLoss(label_smoothing=0.1)   # sampler handles imbalance
optimizer_M = torch.optim.Adam(model_M.parameters(), lr=LR)
scheduler_M = torch.optim.lr_scheduler.StepLR(optimizer_M, step_size=5, gamma=0.5)

model_M, history_M = run_experiment(
    model_M, "M_c_sampler", criterion_M, optimizer_M, scheduler_M, loaders_sampler
)


In [ ]:
# ── Extension N — C architecture + CosineAnnealingLR ─────────────────────────
# StepLR(step=5, gamma=0.5) drops LR abruptly at epochs 5, 10, 15, 20.
# This causes sudden changes in the loss landscape → val_f1 oscillations
# → early stopping may terminate before the model has fully settled.
# CosineAnnealingLR provides a smooth monotone LR decay from LR → eta_min,
# potentially allowing the model to find a flatter minimum and giving
# early stopping a cleaner signal.
# Everything else kept identical to C (LS=0.1, class_weights, Dropout=0.3).
# Expected: similar or slightly better F1 with a smoother training curve.

model_N     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_N = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_N = torch.optim.Adam(model_N.parameters(), lr=LR)
scheduler_N = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_N, T_max=EPOCHS, eta_min=1e-5
)

model_N, history_N = run_experiment(
    model_N, "N_cosine_lr", criterion_N, optimizer_N, scheduler_N, loaders_base
)


---
## Part 3 — Comparison, Final Model & Evaluation

All experiments (A–N + extensions) have now run. This section does four things:

1. **Leaderboard + bar chart** — sort all experiments by `val_macro_f1`, visualise the gaps.
2. **Per-class F1 heatmap** — show every checkpoint's per-class F1 in one grid so you can spot which models are strong in the classes where the best model is weak (useful for ensemble selection).
3. **Soft ensemble** — combine any set of checkpoints at inference time (no retraining); configure the model list below.
4. **Best model deep dive** — reload the winning checkpoint, run the full classification report (per-class F1, precision, recall), and plot training curves + confusion matrix.


In [ ]:
from src.evaluation.analysis import plot_leaderboard

# printed leaderboard table (same as before)
print("=== All Experiments — Sorted by Val Macro-F1 ===\n")
_print_leaderboard(results_tracker)

# bar chart (leaderboard + training time) saved to outputs/plots/
fig = plot_leaderboard(
    results_tracker,
    out_path=TASK_OUT_DIR / "plots" / "task1_leaderboard.png",
)
plt.show()
plt.close(fig)


In [ ]:
from src.evaluation.analysis import plot_per_class_f1_heatmap

# model_registry maps each checkpoint stem to a zero-arg factory function.
# Add or remove entries here when you add new experiments.
# Models NOT listed are skipped silently (e.g. extension checkpoints you haven't defined yet).
model_registry = {
    "A_vanilla":            lambda: VanillaMLP(img_size=IMG_SIZE),
    "B_mlp_base":           lambda: MLP(img_size=IMG_SIZE, dropout=0.4),
    "C_ls01_drop03":        lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
    "D_wd1e4":              lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
    "E_sampler":            lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
    "F_narrow":             lambda: NarrowMLP(img_size=IMG_SIZE, dropout=0.3),
    "G_bottleneck":         lambda: BottleneckMLP(img_size=IMG_SIZE, dropout=0.3),
    "H_vanilla_v2":         lambda: VanillaMLP_v2(img_size=IMG_SIZE),
    "I_v2_rock_weights":    lambda: VanillaMLP_v2(img_size=IMG_SIZE),
    "J_mlp_drop02":         lambda: MLP(img_size=IMG_SIZE, dropout=0.2),
    "K_v2_wd1e5":           lambda: VanillaMLP_v2(img_size=IMG_SIZE),
    # Extension experiments — add after running cells 44-46:
    # "L_wider_ls":         lambda: WiderMLP(img_size=IMG_SIZE, dropout=0.3),
    # "M_c_sampler":        lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
    # "N_cosine_lr":        lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
}

fig = plot_per_class_f1_heatmap(
    checkpoint_dir  = TASK_OUT_DIR / "checkpoints",
    model_registry  = model_registry,
    loader_fn       = build_loaders,           # called fresh for each model
    device          = device,
    out_path        = TASK_OUT_DIR / "plots" / "task1_per_class_f1_heatmap.png",
    highlight_best  = True,                    # adds macro_avg column, sorts rows
)
if fig:
    plt.show()
    plt.close(fig)


In [ ]:
from src.evaluation.ensemble import soft_ensemble, print_ensemble_report

# ── Configure the ensemble here ───────────────────────────────────────────────
# Each entry: (model_instance, checkpoint_path)
# Swap in any two (or more) models from the registry above.
# Current choice: C + E — same architecture, different imbalance mechanisms
# (loss weighting vs data resampling) → complementary errors → genuine diversity.
# ENS_A_G = 0.2257 < C alone = 0.2373: ensembling two weaker models can't beat a
# better individual, so we pair the top-2 models instead.
ensemble_configs = [
    (MLP(img_size=IMG_SIZE, dropout=0.3), TASK_OUT_DIR / "checkpoints" / "C_ls01_drop03.pth"),
    (MLP(img_size=IMG_SIZE, dropout=0.3), TASK_OUT_DIR / "checkpoints" / "E_sampler.pth"),
]

# optional: set weights=[0.6, 0.4] to favour one model; None = uniform average
ensemble_weights = None

# ── Run ───────────────────────────────────────────────────────────────────────
# check all checkpoints exist before attempting
missing = [str(p) for _, p in ensemble_configs if not Path(p).exists()]
if missing:
    print(f"SKIP: missing checkpoints: {missing}")
else:
    ens_result = soft_ensemble(
        checkpoint_configs = ensemble_configs,
        val_loader         = build_loaders()[1],
        device             = device,
        weights            = ensemble_weights,
    )
    print_ensemble_report(ens_result, ensemble_label="C_ls01_drop03 + E_sampler")

    # store in tracker for the leaderboard (key reflects the combination chosen)
    results_tracker["ENS_C_E"] = {
        "val_macro_f1": ens_result["val_macro_f1"],
        "val_acc":      ens_result["val_acc"],
        "val_loss":     float("nan"),
        "total_epochs": 0,
        "train_time_s": 0.0,
        "history":      {},
    }
    _print_leaderboard(results_tracker)


In [ ]:
# ── Load best experiment + full classification report ─────────────────────────
# Pick the experiment with the highest val_macro_f1 automatically.
# All model classes are imported in the shared cell above.

best_name = max(
    (k for k in results_tracker if not k.endswith("_augmented") and k != "ENS_A_G"),
    key=lambda k: results_tracker[k]["val_macro_f1"],
)
best_ckpt = str(TASK_OUT_DIR / "checkpoints" / f"{best_name}.pth")
print(f"Best experiment: {best_name}  (val_macro_f1={results_tracker[best_name]['val_macro_f1']:.4f})")

# rebuild the best model architecture — infer from experiment name
if best_name == "A_vanilla":
    best_model     = VanillaMLP(img_size=IMG_SIZE).to(device)
    best_criterion = nn.CrossEntropyLoss()
elif best_name in ("H_vanilla_v2", "K_v2_wd1e5"):
    best_model     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
    best_criterion = nn.CrossEntropyLoss()
elif best_name == "I_v2_rock_weights":
    weights_best = torch.ones(NUM_CLASSES, device=device)
    weights_best[CLASSES.index("Rock")]   = 3.0
    weights_best[CLASSES.index("Ground")] = 2.0
    best_model     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=weights_best)
elif best_name == "J_mlp_drop02":
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.2).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "E_sampler":
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
elif best_name in ("C_ls01_drop03", "D_wd1e4"):
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "F_narrow":
    best_model     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "G_bottleneck":
    best_model     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:  # B_mlp_base
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights)

best_model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=True))
val_loader_final = build_loaders()[1]  # fresh val loader for final eval

# collect all val predictions for report + confusion matrix
best_model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in val_loader_final:
        imgs = imgs.to(device)
        preds = best_model(imgs).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels_list.extend(labels.tolist())

val_metrics_final = evaluate(best_model, val_loader_final, best_criterion, device)
print(f"\nFinal val_loss={val_metrics_final['loss']:.4f}  val_acc={val_metrics_final['acc']:.4f}  macro_f1={val_metrics_final['macro_f1']:.4f}\n")
print(classification_report_str(all_labels_list, all_preds, CLASSES))


In [ ]:
# ── Training curves + confusion matrix for best experiment ────────────────────
best_history = results_tracker[best_name]["history"]

fig = plot_history(best_history, TASK_OUT_DIR / "plots" / "task1_history.png", title=best_name)
plt.show(); plt.close(fig)

fig = plot_confusion_matrix(all_labels_list, all_preds, CLASSES, TASK_OUT_DIR / "plots" / "task1_confusion.png")
plt.show(); plt.close(fig)


---
## Section 3.1 — Grayscale Experiments

**Motivation:**  
Pokémon type classification is strongly colour-dependent:  
- Fire types are predominantly orange/red  
- Water types are blue  
- Poison types are purple/dark  
- Grass types are green  

Removing colour (converting to grayscale) eliminates the most discriminative pixel channel.
Yet colour comes at a cost: **3× the input dimensionality** (12 288 vs 4 096 features),
which increases the curse of dimensionality on our small dataset.

**Scientific question:** Does the colour information outweigh the curse-of-dimensionality penalty?
We expect **yes** for this dataset — but we test it empirically rather than assume.

**Grayscale pipeline** (implemented in `src/datasets/dataset.py`):
- Input: 64×64 RGB → `Grayscale(1)` → tensor shape `(1, 64, 64)` → flat dim = **4 096**
- Normalisation: `mean=[0.5], std=[0.5]` (standard single-channel norm)
- Optional: `equalize=True` applies `PIL.ImageOps.equalize()` before grayscale — stretches histogram contrast

**Three experiments:**
- **Gray_A:** `VanillaMLP(in_channels=1)` — same architecture as winner A, grayscale input
- **Gray_B:** `MLP(dropout=0.3, in_channels=1)` + equalize — regularised model on equalized gray
- **Gray_C:** `VanillaMLP_v2(in_channels=1)` — wider no-reg model on grayscale (compare with H)


In [ ]:
# ── Experiment Gray_A — VanillaMLP (grayscale, no regularisation) ─────────────
# Same architecture as Exp A (winner) but with in_channels=1.
# Input dimension: 64*64*1 = 4096 (vs RGB's 12288).
# Baseline for the grayscale family — isolates the colour→grayscale effect.

loaders_gray = build_loaders(augment=False, use_sampler=False, grayscale=True, equalize=False)

model_GrayA     = VanillaMLP(img_size=IMG_SIZE, in_channels=1).to(device)
criterion_GrayA = nn.CrossEntropyLoss()
optimizer_GrayA = torch.optim.Adam(model_GrayA.parameters(), lr=LR)
scheduler_GrayA = torch.optim.lr_scheduler.StepLR(optimizer_GrayA, step_size=5, gamma=0.5)

model_GrayA, history_GrayA = run_experiment(
    model_GrayA, "Gray_A_vanilla",
    criterion_GrayA, optimizer_GrayA, scheduler_GrayA, loaders_gray,
)


In [ ]:
# ── Experiment Gray_B — MLP (grayscale + equalize, dropout=0.3) ───────────────
# Adds histogram equalisation before grayscale conversion (stretches contrast).
# Uses regularised MLP to test if equalization + BN/Dropout helps on gray inputs.
# Compare with Gray_A to measure equalization effect, and with D to measure
# the grayscale penalty on the same regularisation level.

loaders_gray_eq = build_loaders(augment=False, use_sampler=False, grayscale=True, equalize=True)

model_GrayB     = MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1).to(device)
criterion_GrayB = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_GrayB = torch.optim.Adam(model_GrayB.parameters(), lr=LR, weight_decay=1e-4)
scheduler_GrayB = torch.optim.lr_scheduler.StepLR(optimizer_GrayB, step_size=5, gamma=0.5)

model_GrayB, history_GrayB = run_experiment(
    model_GrayB, "Gray_B_eq_mlp",
    criterion_GrayB, optimizer_GrayB, scheduler_GrayB, loaders_gray_eq,
)


In [ ]:
# ── Experiment Gray_C — VanillaMLP_v2 (grayscale, no regularisation) ──────────
# Same architecture as Exp H (VanillaMLP_v2) but with in_channels=1.
# Compare: Gray_C vs H → effect of losing colour on the wider vanilla model.
# Compare: Gray_C vs Gray_A → effect of extra width in grayscale regime.

model_GrayC     = VanillaMLP_v2(img_size=IMG_SIZE, in_channels=1).to(device)
criterion_GrayC = nn.CrossEntropyLoss()
optimizer_GrayC = torch.optim.Adam(model_GrayC.parameters(), lr=LR)
scheduler_GrayC = torch.optim.lr_scheduler.StepLR(optimizer_GrayC, step_size=5, gamma=0.5)

model_GrayC, history_GrayC = run_experiment(
    model_GrayC, "Gray_C_v2",
    criterion_GrayC, optimizer_GrayC, scheduler_GrayC, loaders_gray,
)

# --- Summary: RGB vs Grayscale comparison ---
print("\n=== RGB vs Grayscale — Summary ===")
gray_names = ["Gray_A_vanilla", "Gray_B_eq_mlp", "Gray_C_v2"]
rgb_counterparts = {
    "Gray_A_vanilla": "A_vanilla",
    "Gray_B_eq_mlp":  "D_wd1e4",    # closest RGB equivalent (drop=0.3, wd, ls)
    "Gray_C_v2":      "H_vanilla_v2",
}
print(f"\n{'Experiment':<20} {'val_F1':>8}  {'RGB counterpart':<20} {'RGB F1':>8}  {'delta':>7}")
print("-" * 68)
for gname in gray_names:
    if gname not in results_tracker:
        continue
    gf1 = results_tracker[gname]["val_macro_f1"]
    rname = rgb_counterparts[gname]
    rf1 = results_tracker.get(rname, {}).get("val_macro_f1", float("nan"))
    delta_g = gf1 - rf1 if rf1 == rf1 else float("nan")
    print(f"{gname:<20} {gf1:>8.4f}  {rname:<20} {rf1:>8.4f}  {delta_g:>+7.4f}")


---
## Part 4 — Data Augmentation Sanity Check

**Hypothesis:** Data augmentation (`RandomHorizontalFlip` + `ColorJitter`) should NOT meaningfully help an MLP — and we prove it empirically here.

**Why it doesn't help MLP (theoretical):**  
A `RandomHorizontalFlip` swaps pixel position (0, 0) with position (0, W-1). After flattening, the model sees a completely different 12 288-dimensional vector. The MLP has no concept of spatial layout, so a flipped image provides no useful training signal — it looks like a different unrelated example.  
`ColorJitter` adds slight colour variation, which could marginally help by forcing slightly less overfit to exact pixel values, but the effect is small.

**Why we do this anyway:**
1. Empirically confirms the theory — makes the Task 1 → Task 2 story clean: "augmentation doesn't help MLP; it will help CNN."
2. Validates that the augmentation pipeline (`get_augment_transforms`) works end-to-end.
3. Prepares the loader infrastructure for Task 2 and Task 3 where augmentation is critical.

We take the **best experiment configuration** from Part 2 and retrain it with `augment=True`. Same model, same loss, same optimizer — only the training data loader changes.


In [ ]:
# ── Part 4 — Best model + augmentation ───────────────────────────────────────
# Retrain best experiment config with augment=True.
# Only the training loader changes — model, criterion, optimizer are identical.
# Expected: val_macro_f1 roughly equal or slightly worse than without augmentation.

loaders_aug = build_loaders(augment=True, use_sampler=False)

# rebuild a fresh best-model instance with the same architecture and criterion
if best_name == "A_vanilla":
    model_aug     = VanillaMLP(img_size=IMG_SIZE).to(device)
    criterion_aug = nn.CrossEntropyLoss()
elif best_name in ("H_vanilla_v2", "K_v2_wd1e5"):
    model_aug     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
    criterion_aug = nn.CrossEntropyLoss()
elif best_name == "I_v2_rock_weights":
    weights_aug = torch.ones(NUM_CLASSES, device=device)
    weights_aug[CLASSES.index("Rock")]   = 3.0
    weights_aug[CLASSES.index("Ground")] = 2.0
    model_aug     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=weights_aug)
elif best_name == "J_mlp_drop02":
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.2).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "E_sampler":
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(label_smoothing=0.1)
elif best_name in ("C_ls01_drop03", "D_wd1e4"):
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "F_narrow":
    model_aug     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "G_bottleneck":
    model_aug     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:  # B_mlp_base
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights)

optimizer_aug = torch.optim.Adam(model_aug.parameters(), lr=LR, weight_decay=1e-4)
scheduler_aug = torch.optim.lr_scheduler.StepLR(optimizer_aug, step_size=5, gamma=0.5)

model_aug, history_aug = run_experiment(
    model_aug, f"{best_name}_augmented",
    criterion_aug, optimizer_aug, scheduler_aug, loaders_aug,
)

# compare augmented vs non-augmented side by side
f1_no_aug = results_tracker[best_name]["val_macro_f1"]
f1_aug    = results_tracker[f"{best_name}_augmented"]["val_macro_f1"]
delta     = f1_aug - f1_no_aug

print(f"\n=== Augmentation effect on best model ({best_name}) ===")
print(f"  Without augmentation : val_macro_f1 = {f1_no_aug:.4f}")
print(f"  With augmentation    : val_macro_f1 = {f1_aug:.4f}")
print(f"  Delta                : {delta:+.4f}  ({'improved' if delta > 0.005 else 'no meaningful gain' if abs(delta) <= 0.005 else 'slightly hurt'})")
print(f"\nConclusion: augmentation {'helps' if delta > 0.005 else 'does not meaningfully help'} MLP on this dataset.")
print("This is expected: MLP flattens the image, so a flipped image is an unrelated vector.")
print("Augmentation will be valuable for CNN (Task 2) where spatial structure is preserved.")


In [ ]:
# ── Save all results to disk ──────────────────────────────────────────────────
# task1_results.json contains everything needed for the report — download it from Colab.
from sklearn.metrics import f1_score as _sk_f1

per_class_f1_arr = _sk_f1(all_labels_list, all_preds, average=None, zero_division=0)

# total time across all experiments
total_experiment_time = sum(v["train_time_s"] for v in results_tracker.values())

output = {
    "best_experiment":    best_name,
    "val_macro_f1":       round(val_metrics_final["macro_f1"], 4),
    "val_accuracy":       round(val_metrics_final["acc"], 4),
    "val_loss":           round(val_metrics_final["loss"], 4),
    "per_class_f1":       {cls: round(float(per_class_f1_arr[i]), 4) for i, cls in enumerate(CLASSES)},
    "all_experiments":    {
        name: {k: v for k, v in m.items() if k != "history"}
        for name, m in results_tracker.items()
    },
    "total_experiment_time_s": round(total_experiment_time, 1),
    "config": {
        "FAST_RUN": FAST_RUN, "EPOCHS": EPOCHS, "LR": LR,
        "BATCH_SIZE": BATCH_SIZE, "PATIENCE": PATIENCE, "IMG_SIZE": IMG_SIZE,
    },
    "best_history": {k: [round(v, 4) for v in vals] for k, vals in best_history.items()},
}

results_path = TASK_OUT_DIR / "results" / "task1_results.json"
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved -> {results_path}")
print(f"\nBest: {best_name}  val_macro_f1={output['val_macro_f1']}  val_acc={output['val_accuracy']}")
print(f"Total experiment time: {total_experiment_time:.0f}s across {len(results_tracker)} experiments")
print(json.dumps({"per_class_f1": output["per_class_f1"]}, indent=2))


In [ ]:
# ── Generate + validate submission CSV ───────────────────────────────────────
from src.evaluation.submission import generate_submission, validate_submission

SUB_PATH = TASK_OUT_DIR / "results" / "submission_task1.csv"
generate_submission(best_model, test_loader, CLASSES, SUB_PATH, device)
validate_submission(SUB_PATH, expected_rows=900)
print(f"Submission saved -> {SUB_PATH}")


In [ ]:
# ── Download outputs zip (Colab only) ────────────────────────────────────────
# Run this cell after all experiments to get plots, checkpoints, and results JSON.
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = Path(ROOT) / "task1_outputs"
    shutil.make_archive(str(zip_path), "zip", root_dir=str(TASK_OUT_DIR.parent), base_dir=TASK_OUT_DIR.name)
    files.download(str(zip_path) + ".zip")
else:
    print(f"Outputs at: {TASK_OUT_DIR.resolve()}")


---
## Summary

### Run results (fill in after full Colab run)

| Metric | Value |
|---|---|
| Best experiment | _(fill after run)_ |
| Val macro-F1 (best) | _(fill)_ |
| Val accuracy (best) | _(fill)_ |
| Kaggle public score | _(fill after submission)_ |
| Epochs run (best exp) | _(fill)_ |
| Total experiment time | _(fill)_ |
| Best per-class F1 | _(fill — likely Fire or Water)_ |
| Worst per-class F1 | _(fill — likely Rock or Ground)_ |

### Experiments run

| ID | Name | Architecture | Monitor | Notes |
|---|---|---|---|---|
| A | vanilla | VanillaMLP (128→64), no BN/Drop | val_macro_f1 | Bare baseline |
| B | mlp_base | MLP (512→256→128), Drop=0.4 | val_macro_f1 | First Colab run equivalent |
| C | ls01_drop03 | MLP, Drop=0.3 | val_macro_f1 | + label_smoothing=0.1 |
| D | wd1e4 | MLP, Drop=0.3 | val_macro_f1 | + weight_decay=1e-4 |
| E | sampler | MLP, Drop=0.3 | val_macro_f1 | WeightedRandomSampler |
| F | narrow | NarrowMLP (256→128→64→32) | val_macro_f1 | Deeper + narrow |
| G | bottleneck | BottleneckMLP (512→1024→256→128) | val_macro_f1 | Wide middle |
| H | vanilla_v2 | VanillaMLP_v2 (256→128), no reg | val_macro_f1 | More capacity, no reg |
| I | v2_rock_weights | VanillaMLP_v2, CE(Rock×3, Ground×2) | val_macro_f1 | Targeted minority upweighting |
| J | mlp_drop02 | MLP, Drop=0.2 | val_macro_f1 | Lighter dropout |
| K | v2_wd1e5 | VanillaMLP_v2, WD=1e-5 | val_macro_f1 | Minimal L2 boundary test |
| ENS_A_G | Ensemble A+G | Soft avg (A_vanilla + G_bottleneck) | — | Inference-time only |
| Gray_A | vanilla (gray) | VanillaMLP(in_channels=1) | val_macro_f1 | Grayscale, no reg |
| Gray_B | eq_mlp (gray+eq) | MLP(drop=0.3, in_channels=1) | val_macro_f1 | Grayscale + histogram equalize |
| Gray_C | v2 (gray) | VanillaMLP_v2(in_channels=1) | val_macro_f1 | Wider vanilla on grayscale |
| _best_aug | augmented | Best arch + augment=True | val_macro_f1 | Expected: no meaningful gain |

### Expected findings (confirm after run)

1. **Regularisation effect:** simpler models (A, H) are expected to score close to or above the regularised ones — the dataset is small enough that over-regularisation hurts as much as under-regularisation. Verify by comparing A vs B vs D in the leaderboard.
2. **Augmentation effect:** augmentation is expected to hurt or have no effect on MLP. `RandomHorizontalFlip` produces a completely different 12 288-vector — the model has no spatial bias to benefit from it. Part 4 confirms this empirically.
3. **Grayscale effect:** grayscale experiments are expected to score lower than their RGB counterparts — colour is the primary discriminative signal (Fire=orange, Water=blue). Gray_B (with equalization) may partially recover on dark/washed-out sprites.
4. **Hard classes:** Rock and Ground are expected to have the lowest per-class F1. Their colour profiles (grey/brown) overlap with each other and with Fighting/Normal. The confusion matrix should show Rock and Ground frequently predicted as each other.
5. **MLP ceiling:** val macro-F1 is expected to plateau around 0.20–0.25 regardless of regularisation choices. The fundamental bottleneck is lack of spatial structure — CNN (Task 2) will exceed this by preserving pixel neighbourhood relationships.

### Infrastructure built (`src/`)

- `src/models/mlp.py` — 5 MLP classes with `in_channels` param (RGB=3 or grayscale=1)
- `src/datasets/dataset.py` — RGB, augmented, grayscale, and gray+equalize transform pipelines; `grayscale`/`equalize` params in loaders
- `src/training/early_stopping.py` — metric-agnostic, monitors by minimisation (`stopper(-val_f1, model)`)
- `src/evaluation/metrics.py`, `plots.py`, `submission.py` — classification report, training curves, confusion matrix, submission CSV
- Tests: `models_test.py` (10), `dataset_test.py` (13), `train_test.py` (3) — all pass locally
